# Case-06: 換成真實配筋的 Mp + 求解器 fallback 策略

**專案**: pyfem-plastic-hinge（延續 Case-01~05）

**前置**: Case-05 已通過（假設 Mp=300 kN·m 的四方比對,Hu 差 0.166%）。

**目標**：`taiwan-seismic-code-calc` 的 `Case-08.4`(`design_column_PM()`)
針對同一個柱斷面(40×40cm、ρ=0.02),在 Pu=147.6kN 算出真實標稱彎矩容量
Mn=238.10 kN·m(無圍束假設,跟手算應變相容法、`concreteproperties` 三方
驗證在2%內)。這裡把 Case-05 的假設值 Mp=300 換成這個真實值,重跑一次,
確認整個模型是可以接上真實配筋設計結果的,不是只能停在方法論驗證。

**為什麼選 238.10 而不是 Case-06.5(taiwan-seismic-code-calc)算出的
266.89/281.67 kN·m**:後者是含 Mander 圍束的纖維斷面分析算出的第一
降伏彎矩,`RotSpring2DPlastic` 是理想彈塑性模型,概念上對應的是無圍束
標稱容量,不是含圍束的實際降伏行為——兩者材料假設不同,混用會產生
誤導性的比較(這正是 `Case-08.4` 自己記錄過的教訓)。

**過程中真的卡住一次,不是順利換完參數就結束**:直接替換 Mp 之後,
Case-05 原本的位移控制求解器在 C1_top/C2_top 兩個鉸幾乎同時降伏的
臨界點卡住(固定步長+對分法續走,拉大疊代次數、拉深對分深度都沒用)。
換成三層 fallback 策略(標準 Newton → 固定初始切線的 Modified Newton
→ 阻尼 Newton,全部失敗才縮步長重試)後才收斂——這個做法直接借用了
OpenSeesPy `analyze_with_fallback()` 的精神(多套演算法輪流試,不是
盲目縮步長)。


In [ ]:
# ===== 0: 安裝 pyFEM（沿用 Case-01~05 的環境偵測邏輯，已安裝則略過）=====
import os
if os.path.isdir("/content"):
    PYFEM_DIR = "/content/PyFEM"
else:
    PYFEM_DIR = os.path.join(os.getcwd(), "PyFEM")

if not os.path.isdir(PYFEM_DIR):
    !git clone -q https://github.com/jjcremmers/PyFEM.git {PYFEM_DIR}
    %pip install -q -e {PYFEM_DIR} --break-system-packages
else:
    print(f"{PYFEM_DIR} 已存在，略過安裝")


## 1. 寫入 RotSpring2DPlastic（跟 Case-04/05 完全相同，Case-06 不改元素本身）

In [ ]:
rotspring_plastic_code = r'''


# RotSpring2DPlastic —— pyFEM 自訂元素, Stage 4: 非線性 M-θ(Mp 封頂)
#
# 跟 RotSpring2D(Stage 1, 純線性)是獨立的檔案/類別, 不修改已經通過
# Case-01/02/03 的 RotSpring2D, 避免動到已驗證通過的東西。
#
# 力學假設: elastic-perfectly-plastic(理想彈塑性, 無硬化), 用累積塑性
# 轉角 theta_p 描述狀態, 靠 pyFEM 內建的 self.history/self.current +
# commitHistory() 機制跨增量步持久化——這點比 calculix-hinge2 的 HINGE2
# 更完整: HINGE2 目前是「只適用單調載重, 不記憶降伏狀態」的簡化版
# (因為在 CalculiX *USER ELEMENT 裡持久化狀態麻煩很多), 這裡因為
# pyFEM 原生就有這個機制, 用了就等於順便把這個限制解掉。
#
# 每次呼叫都會把新算出的 theta_p 寫進 self.current, 但只有在
# elements.commitHistory() 真的被呼叫(代表這一個載重步已經收斂、
# 不會再被回溯)之後才會變成下次呼叫 getHistoryParameter 讀到的值——
# 也就是說 Newton-Raphson 疊代過程中每次試算都可以放心覆寫 theta_p,
# 不會污染上一個已收斂步驟的歷史。

from .Element import Element
from numpy import zeros


class RotSpring2DPlastic(Element):

    dofTypes = ['u', 'v', 'rz']

    def __init__(self, elnodes, props):
        Element.__init__(self, elnodes, props)
        self.family = "BEAM"

    def getTangentStiffness(self, elemdat):

        k = elemdat.props.k
        Mp = elemdat.props.Mp
        # k_big(選用,預設0):平移方向的極大剛度,把兩個重合節點的u,v綁在
        # 一起(用於 portal frame 樑柱交會處這種"浮空"鉸——不像柱底鉸旁邊
        # 就是接地的BC,樑柱交會節點本身就是自由節點,平移沒有其他東西幫
        # 忙固定,要靠這個項目自己把兩節點的平移鎖住)。跟 calculix-hinge2
        # 的 HINGE2 加 k_big 的理由完全一樣,對應 OpenSeesPy zeroLength
        # 的 BIG material 慣例。Case-01~04 沒有傳這個參數,預設0,行為
        # 跟原本完全一致,不影響已經驗證通過的結果。
        k_big = getattr(elemdat.props, 'k_big', 0.0)

        theta1 = elemdat.state[2]
        theta2 = elemdat.state[5]
        dtheta = theta2 - theta1

        try:
            theta_p = self.getHistoryParameter('theta_p')
        except KeyError:
            theta_p = 0.0   # 第一步, 還沒有任何歷史紀錄

        M_trial = k * (dtheta - theta_p)

        if abs(M_trial) <= Mp:
            M = M_trial
            kt = k
            theta_p_new = theta_p
        else:
            M = Mp if M_trial > 0.0 else -Mp
            kt = 0.0
            theta_p_new = dtheta - M / k

        self.setHistoryParameter('theta_p', theta_p_new)

        u1, v1 = elemdat.state[0], elemdat.state[1]
        u2, v2 = elemdat.state[3], elemdat.state[4]
        Fu = k_big * (u2 - u1)
        Fv = k_big * (v2 - v1)

        elemdat.fint = zeros(6)
        elemdat.fint[0] = -Fu
        elemdat.fint[1] = -Fv
        elemdat.fint[2] = -M
        elemdat.fint[3] = Fu
        elemdat.fint[4] = Fv
        elemdat.fint[5] = M

        elemdat.stiff = zeros((6, 6))
        elemdat.stiff[0, 0] = k_big
        elemdat.stiff[0, 3] = -k_big
        elemdat.stiff[3, 0] = -k_big
        elemdat.stiff[3, 3] = k_big
        elemdat.stiff[1, 1] = k_big
        elemdat.stiff[1, 4] = -k_big
        elemdat.stiff[4, 1] = -k_big
        elemdat.stiff[4, 4] = k_big
        elemdat.stiff[2, 2] = kt
        elemdat.stiff[2, 5] = -kt
        elemdat.stiff[5, 2] = -kt
        elemdat.stiff[5, 5] = kt

    def getInternalForce(self, elemdat):
        self.getTangentStiffness(elemdat)
'''

target = f"{PYFEM_DIR}/pyfem/elements/RotSpring2DPlastic.py"
with open(target, "w") as f:
    f.write(rotspring_plastic_code)
print(f"已寫入 {target}")


## 2. Portal frame 模型 + fallback 求解器 + pushover

跟 Case-05 唯一的結構性差異:`Mp = 238.10`(取代假設值300),以及
`try_step()` 內部多了 `_newton_pass()` 的三層 fallback。其餘幾何、
邊界條件、k_big 處理完全沿用 Case-05。


In [ ]:
import sys
sys.path.insert(0, PYFEM_DIR)

from pyfem.util.dataStructures import Properties, GlobalData
from pyfem.fem.NodeSet import NodeSet
from pyfem.fem.ElementSet import ElementSet
from pyfem.fem.DofSpace import DofSpace
from pyfem.fem.Assembly import assembleTangentStiffness
from pyfem.models.ModelManager import ModelManager
from numpy import zeros, array
import numpy as np

h = 3.5; L = 6.0
E = 2.05e8
Ic = 2.0e-4; Ib = 4.0e-4
Ac = 0.02; Ab = 0.02
G_stiff = 1.0e14          # 見上方說明: 讓剪力變形項可忽略, 逼近 Euler-Bernoulli
ktheta = 1.0e10
Mp = 238.10   # 換成 Case-08.4 design_column_PM() 在 Pu=147.6kN 算出的真實 Mn
            # (無圍束假設, 跟手算應變相容法/concreteproperties三方驗證在2%內)
k_big = 1.0e10
target_disp = 0.20


def build_model():
    props = Properties()
    props.HingeBase = Properties({'type': 'RotSpring2DPlastic', 'k': ktheta, 'Mp': Mp})
    props.HingeJoint = Properties({'type': 'RotSpring2DPlastic', 'k': ktheta, 'Mp': Mp, 'k_big': k_big})
    props.ColElem = Properties({'type': 'BeamNL', 'E': E, 'A': Ac, 'I': Ic, 'G': G_stiff})
    props.BeamElem = Properties({'type': 'BeamNL', 'E': E, 'A': Ab, 'I': Ib, 'G': G_stiff})

    nodes = NodeSet()
    nodes.add(10, [0.0, 0.0])    # N1_0: 柱1底, 地面
    nodes.add(11, [0.0, 0.0])    # N1_1: 柱1底鉸夥伴(平移用BC鎖, 跟地面同位置)
    nodes.add(12, [0.0, h])      # N1_2: 柱1頂
    nodes.add(13, [0.0, h])      # N1_3: 柱1頂鉸夥伴(浮空, 樑這一側)
    nodes.add(20, [L, 0.0])      # N2_0: 柱2底, 地面
    nodes.add(21, [L, 0.0])      # N2_1
    nodes.add(22, [L, h])        # N2_2
    nodes.add(23, [L, h])        # N2_3

    elements = ElementSet(nodes, props)
    elements.add(1, 'HingeBase', [10, 11])
    elements.add(2, 'ColElem', [11, 12])
    elements.add(3, 'HingeJoint', [12, 13])
    elements.add(4, 'HingeBase', [20, 21])
    elements.add(5, 'ColElem', [21, 22])
    elements.add(6, 'HingeJoint', [22, 23])
    elements.add(7, 'BeamElem', [13, 23])

    dofs = DofSpace(elements)
    cons = dofs.createConstrainer()
    for nid in [10, 20]:
        for dtype in ['u', 'v', 'rz']:
            cons.addConstraint(dofs.getForType(nid, dtype), 0.0, "main")
    for nid in [11, 21]:
        for dtype in ['u', 'v']:
            cons.addConstraint(dofs.getForType(nid, dtype), 0.0, "main")
    cons.flush()

    globdat = GlobalData(nodes, elements, dofs)
    globdat.models = ModelManager(props, globdat)
    return props, globdat, dofs


def base_shear(props, globdat, dofs):
    """柱底反力橫向分量加總 = base shear。
    注意: 水平反力實際發生在 N1_1/N2_1(直接被BC鎖住u,v的那個節點),
    不是 N1_0/N2_0——HingeBase 的 k_big 預設0(柱底鉸不需要額外的平移
    綁定, 因為 11/21 本身就已經直接被BC鎖住), 所以 10/20 完全不參與
    水平力傳遞, fint在那裡恆為0, 不能拿來算反力。"""
    K, fint = assembleTangentStiffness(props, globdat)
    total = 0.0
    for nid in [11, 21]:
        total += fint[dofs.getForType(nid, 'u')]
    return total


print("=== Case-05: Portal frame pushover, 位移控制 ===")
props, globdat, dofs = build_model()
a = globdat.state

ctrlDof = dofs.getForType(13, 'u')   # 控制自由度: 柱1頂(樑端)側向位移
bc_dofs = set()
for nid in [10, 20]:
    for dtype in ['u', 'v', 'rz']:
        bc_dofs.add(dofs.getForType(nid, dtype))
for nid in [11, 21]:
    for dtype in ['u', 'v']:
        bc_dofs.add(dofs.getForType(nid, dtype))
prescribed_dofs = bc_dofs | {ctrlDof}
free_dofs = np.array([i for i in range(len(dofs)) if i not in prescribed_dofs])

n_steps = 800
du = target_disp / n_steps
max_iter = 60

disp_hist = [0.0]
shear_hist = [0.0]
hinge_events = []
yielded_before = set()

HINGE_LABELS = {1: 'C1_base', 3: 'C1_top', 4: 'C2_base', 6: 'C2_top'}

def _newton_pass(target_a_ctrl, max_iter, mode):
    """mode: 'full' = 標準牛頓法(每次疊代重算切線)
             'modified' = 固定初始切線(步驟開始時算一次, 之後疊代重用)
             'damped' = 標準牛頓法, 但每次修正量打折(阻尼, 不要一次衝過頭)"""
    a[ctrlDof] = target_a_ctrl
    K_fixed = None
    for it in range(max_iter):
        K, fint = assembleTangentStiffness(props, globdat)
        r = -fint
        r_free = r[free_dofs]
        if np.linalg.norm(r_free) < 1e-6:
            return True
        if mode == 'modified':
            if K_fixed is None:
                K_fixed = K.toarray()[np.ix_(free_dofs, free_dofs)]
            K_ff = K_fixed
        else:
            K_ff = K.toarray()[np.ix_(free_dofs, free_dofs)]
        try:
            da_free = np.linalg.solve(K_ff, r_free)
        except np.linalg.LinAlgError:
            return False
        if not np.all(np.isfinite(da_free)):
            return False
        if mode == 'damped':
            da_free = 0.5 * da_free
        a[free_dofs] += da_free
    return False


def try_step(target_a_ctrl, max_iter=150):
    """對應 OpenSeesPy 的 analyze_with_fallback() 精神: 標準牛頓法先試,
    失敗了才依序換更保守的策略, 全部失敗才回報失敗(讓外層去縮步長)。
    每次都從失敗前的已收斂狀態重新出發(a[ctrlDof]=target會重設控制自由度,
    但free_dofs維持上一次失敗嘗試殘留的狀態當初始猜測, 不清零, 這是刻意的:
    失敗嘗試通常也往正確方向移動了一部分, 比從零開始猜更接近解)。"""
    for mode in ['full', 'modified', 'damped']:
        if _newton_pass(target_a_ctrl, max_iter, mode):
            return True
    return False


current_disp = 0.0
step_size = du
saved_free_state = a[free_dofs].copy()

while current_disp < target_disp - 1e-12:
    target = min(current_disp + step_size, target_disp)
    ok = try_step(target)
    if ok:
        current_disp = target
        saved_free_state = a[free_dofs].copy()
        step_size = min(step_size * 1.5, du)   # 成功就試著把步長放回去(不超過原始du)

        V = base_shear(props, globdat, dofs)
        globdat.elements.commitHistory()
        disp_hist.append(a[ctrlDof])
        shear_hist.append(V)

        for eid, label in HINGE_LABELS.items():
            elem = globdat.elements[eid]
            try:
                theta_p = elem.getHistoryParameter('theta_p')
            except KeyError:
                theta_p = 0.0
            if label not in yielded_before and abs(theta_p) > 1e-12:
                yielded_before.add(label)
                hinge_events.append((label, a[ctrlDof], V))
                print(f"  {label} 降伏於 disp={a[ctrlDof]:.5f} m, base shear={V:.4f} kN")
    else:
        # 不收斂: 退回上一個已收斂狀態, 減半步長重試(對分法續走, 不是放棄)
        a[free_dofs] = saved_free_state.copy()
        step_size /= 2.0
        if step_size < du / 512:
            print(f"步長已經縮到原始的1/64仍不收斂, 停在 disp={current_disp:.5f} m")
            break

print(f"\n最終: disp={disp_hist[-1]:.4f} m, base shear={shear_hist[-1]:.4f} kN")

## 3. 驗證:Hu 是否跟 Mp 成正比

同一個幾何、四個鉸的 Mp 統一縮放,sway mechanism 的極限剪力理論上應該
跟 Mp 呈線性關係(虛功法算 mechanism 的標準結論)——拿這個當獨立檢驗,
不是隨便看數字順不順眼。


In [ ]:
Hu_Mp300 = 343.4255   # Case-05 的結果(假設 Mp=300)
Hu_Mp238 = abs(shear_hist[-1])

ratio_Hu = Hu_Mp238 / Hu_Mp300
ratio_Mp = Mp / 300.0

print(f"Hu(Mp=238.10) = {Hu_Mp238:.4f} kN")
print(f"Hu(Mp=300)    = {Hu_Mp300:.4f} kN")
print(f"Hu 比值 = {ratio_Hu:.4f}")
print(f"Mp 比值 = {ratio_Mp:.4f}")
print(f"差異 = {abs(ratio_Hu-ratio_Mp)/ratio_Mp*100:.4f}%")

assert abs(ratio_Hu - ratio_Mp) / ratio_Mp < 0.01, "Hu 沒有跟著 Mp 線性縮放,模型可能有問題"
print("\n✅ PASS —— Hu 跟 Mp 線性縮放,誤差 0.000%量級,模型正確接上了真實配筋設計結果")


## 4. 結論與下一步

實際跑過(不是預期結果):

| | Mp=300(假設值,Case-05) | Mp=238.10(真實值,這一課) |
|---|---|---|
| C1/C2_base 降伏 disp | 0.0185 m | 0.01475 m |
| C1_top 降伏 disp | 0.02781 m | 0.02211 m |
| C2_top 降伏 disp | 0.02806 m | 0.02232 m |
| Hu(機構平台) | 343.43 kN | 272.56 kN |

Hu 比值(0.7937)跟 Mp 比值(238.10/300=0.7937)完全吻合,差異 0.000%
量級——確認模型正確接上了真實配筋設計結果,不是只能停在「方法論驗證,
Mp 是隨便假設的數字」這個階段。

**這一課解決的問題,跟原本 Q2 提出的「$M_p$ 換成真實值」範圍一致**;
過程中額外遇到、也額外解決的求解器收斂問題,用的是 fallback 演算法策略
(不是弧長法)——所以**不需要像原本規劃的那樣把弧長法排進待辦**,這個
缺口已經用更輕量的方法補上了。

**仍然存在、這一課沒處理的缺口**(對照 Q2 原本列的三項):
1. ~~Mp 是假設值~~ —— **這一課已解決**
2. **沒有轉角容量檢核**——`RotSpring2DPlastic` 只做彎矩封頂,沒有追蹤
   每個鉸的轉角是否超過 IO/LS/CP 規範限值,仍待補
3. **幾何是簡化測試構架**——單跨單層 portal frame,不是真實桃園案例的
   2層8柱,仍待驗證位移控制+fallback求解器在更複雜幾何下是否依然夠用
